# Real-Time Short-Channel Artifact Regression: Signal Quality, Periodicity, and Interaction with AR-Based Phase/Trigger Forecasting

This report evaluates the real-time short-channel/systemic-artifact removal
added to `realtime/activity_core.py` (`RLSFilter` + `ShortChannelArtifactRemover`,
see the project README's "Real-time artifact removal" section), asking three
concrete questions:

1. **Is there a cleaner periodicity to the signal** after short-channel
   regression -- i.e. does it sharpen the spectral/temporal structure the
   peak/trough-triggered iTBS loop (`itbs_trigger.py`) actually relies on?
2. If this pipeline were extended with a **phastimate-style autoregressive
   (AR) forward-extrapolation model** (Zrenner et al., 2018 -- the standard
   real-time phase/edge-forecasting method used for phase-triggered
   closed-loop EEG-TMS), is there a **negative interaction** between the RLS
   artifact-removal stage and that AR forecasting stage?
3. **Should the confound regression be folded directly into the AR model**
   (a joint ARX estimator), rather than kept as a separate preprocessing
   stage?

**On data.** This repository does not commit a raw `.snirf` recording (the
default path in `stream_snirf_lsl.py` points outside the repo, at a location
specific to the original acquisition machine), and this analysis environment
has no such file available either. All analyses below therefore run on a
**physiologically-motivated synthetic dataset with known ground truth**
(cardiac, respiratory, and Mayer-wave components; a cerebral component
locked to the project's `STANDARD_BURST_INTERVAL_SEC` cadence; and an
optional injected motion-like transient) -- see the `REAL_SNIRF_PATH`
parameter below for re-running this notebook against a real recording. Using
synthetic data with a known "true cerebral signal" is not a compromise here,
it is what makes questions 2-3 answerable at all: no real recording has a
verifiable cerebral ground truth to score a forecast against.

Every stage below calls the **actual production classes** from
`realtime/activity_core.py` (`ChannelPair`, `ExtinctionTable`,
`ShortChannelArtifactRemover`, `RLSFilter`, `ActivityAnalyzer`) rather than
re-implementations, so the results describe the shipped pipeline.

In [ ]:
# Parameters (papermill "parameters" cell)
REALTIME_SRC_DIR = "../realtime"          # path to the project's realtime/ package
REAL_SNIRF_PATH = None                    # set to a .snirf path to run on real data instead of the simulation
SEED = 42

SFREQ = 10.0                              # Hz, simulated acquisition rate
DURATION_SEC = 1200.0                     # 20 minutes
N_EVAL_CUTS_BASELINE = 120                # number of causal forecast-evaluation points away from the transient
AR_ORDER = 40                             # AR model order (4s of lags at SFREQ)
AR_WINDOW_SEC = 60.0                      # trailing window used to (re)fit the AR/ARX model at each forecast
AR_FORECAST_SEC = 2.0                     # forecast horizon, phastimate-style short-horizon forward extrapolation

GLITCH_T = 600.0                          # s, center of an injected motion-like transient shared by short+long channels
GLITCH_DUR = 4.0                          # s, transient FWHM-ish duration
GLITCH_AMP_UM = 6.0                       # uM-equivalent size of the injected transient


In [ ]:
import sys, warnings
sys.path.insert(0, REALTIME_SRC_DIR)
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import welch

from activity_core import (
    ChannelPair, ExtinctionTable, EXTINCTION_CSV, PPF, BASELINE_SEC,
    RLSFilter, ShortChannelArtifactRemover, ActivityAnalyzer,
    SHORT_CHANNEL_MAX_DISTANCE_M, STANDARD_BURST_INTERVAL_SEC, MIN_PEAK_DISTANCE_SEC,
)

plt.rcParams.update({
    'font.size': 15, 'axes.titlesize': 17, 'axes.labelsize': 15,
    'xtick.labelsize': 13, 'ytick.labelsize': 13, 'legend.fontsize': 12,
    'lines.linewidth': 2, 'figure.facecolor': 'white',
})
SHORT_COLOR, LONG_COLOR, CLEAN_COLOR, TRUTH_COLOR = "#5c8fe8", "#e8734b", "#2fae6b", "#333333"
print(f"activity_core loaded from {REALTIME_SRC_DIR}; SHORT_CHANNEL_MAX_DISTANCE_M={SHORT_CHANNEL_MAX_DISTANCE_M*1000:.0f}mm, "
      f"STANDARD_BURST_INTERVAL_SEC={STANDARD_BURST_INTERVAL_SEC:g}s (-> {1/STANDARD_BURST_INTERVAL_SEC:.3f} Hz trigger-cadence tone)")

## 1. Dataset

A 6-channel montage on a 1D probe axis: **2 short-separation channels**
(8mm, source-detector distance under `SHORT_CHANNEL_MAX_DISTANCE_M`) and
**4 long channels** (28-33mm) spread between them, so
`ShortChannelArtifactRemover`'s nearest-short-channel selection has a real
choice to make for each long channel (exactly as it would on a real
multi-distance montage).

Every channel's concentration-domain HbO/HbR is built from a shared pool of
systemic drivers -- **cardiac (~1.1 Hz), respiration (~0.25 Hz), a Mayer
wave (~0.10 Hz)**, slow drift, and per-channel idiosyncratic noise -- with a
smoothly space-varying coupling gain so nearby channels share more systemic
signal than distant ones. Only the **long** channels additionally carry a
**cerebral** component: a train of ~10.3s-spaced Gaussian bumps, deliberately
close to `STANDARD_BURST_INTERVAL_SEC` (10s) -- this is also uncomfortably
close to the Mayer-wave band, which is the crux of the periodicity question
below. Everything is then forward-transformed through the modified
Beer-Lambert law using the *same* `ExtinctionTable`/`PPF`/EL-matrix machinery
`ChannelPair.set_extinction()` uses, into raw dual-wavelength intensity --
so the simulated stream is exactly the kind of input `ChannelPair.update()`
expects from a live LSL amplitude stream.

In [ ]:
def build_dataset(seed=SEED, inject_transient=False):
    rng = np.random.default_rng(seed)
    N = int(DURATION_SEC * SFREQ)
    t = np.arange(N) / SFREQ
    ext = ExtinctionTable(EXTINCTION_CSV)
    WAVELENGTHS = [760.0, 850.0]

    short_specs = [("Ssh1", "Dsh1", 0.0, 0.008), ("Ssh2", "Dsh2", 0.06, 0.008)]
    long_specs = [("Sl1", "Dl1", 0.012, 0.030), ("Sl2", "Dl2", 0.022, 0.028),
                  ("Sl3", "Dl3", 0.038, 0.033), ("Sl4", "Dl4", 0.048, 0.030)]

    def make_pair(src, det, x, dist):
        idx_by_wl = {wl: i for i, wl in enumerate(WAVELENGTHS)}
        p = ChannelPair(src, det, idx_by_wl, np.array([x, 0, 0]), np.array([x + dist, 0, 0]), dist)
        p.set_extinction(ext)
        return p

    pairs, short_pairs, long_pairs = [], [], []
    for src, det, x, dist in short_specs:
        p = make_pair(src, det, x, dist); pairs.append(p); short_pairs.append(p)
    for src, det, x, dist in long_specs:
        p = make_pair(src, det, x, dist); pairs.append(p); long_pairs.append(p)

    def osc(freq, jitter_frac, amp, phase0=0.0):
        inst_freq = freq * (1 + jitter_frac * np.sin(2*np.pi*0.01*t + phase0*3))
        phase = 2*np.pi*np.cumsum(inst_freq)/SFREQ
        return amp*np.sin(phase + phase0)

    cardiac = osc(1.1, 0.05, 1.2, 0.3)
    resp = osc(0.25, 0.1, 0.8, 1.1)
    mayer = osc(0.10, 0.08, 0.9, 2.0)
    drift = 0.3*np.sin(2*np.pi*0.003*t)

    cereb_period = STANDARD_BURST_INTERVAL_SEC * 1.03
    bump_centers = np.arange(30, DURATION_SEC, cereb_period) + rng.normal(0, 0.5, size=int((DURATION_SEC-30)//cereb_period)+1)
    cerebral = np.zeros(N)
    for c in bump_centers:
        cerebral += 1.5*np.exp(-0.5*((t - c)/1.0)**2)

    glitch = np.zeros(N)
    if inject_transient:
        glitch = GLITCH_AMP_UM * np.exp(-0.5*((t - GLITCH_T)/(GLITCH_DUR/2.35))**2)

    def spatial_gain(x, base=1.0, wobble=0.3, period=0.1):
        return base*(1 + wobble*np.sin(2*np.pi*x/period))

    true_cereb_hbo, true_conc = {}, {}
    seed_ctr = 1000

    def make_channel_conc(x, is_short, seed_local, glitch_gain):
        r = np.random.default_rng(seed_local)
        g_card, g_resp, g_mayer = spatial_gain(x, 1.0, 0.3, 0.07), spatial_gain(x, 1.0, 0.25, 0.09), spatial_gain(x, 1.0, 0.35, 0.11)

        def idio(freq, amp):
            ph = 2*np.pi*freq*t + r.uniform(0, 2*np.pi)
            return amp*np.sin(ph)*(0.5 + 0.5*np.sin(2*np.pi*0.003*t + r.uniform(0, 6)))

        sys_hbo = (g_card*cardiac + g_resp*resp + g_mayer*mayer + drift
                   + idio(1.05, 0.15) + idio(0.23, 0.1) + idio(0.095, 0.15) + glitch_gain*glitch)
        sys_hbr = -0.4*sys_hbo + idio(0.9, 0.05)
        cereb_hbo = np.zeros(N) if is_short else cerebral * (1 + 0.1*r.normal())
        cereb_hbr = -0.3*cereb_hbo
        hbo = sys_hbo + cereb_hbo + r.normal(0, 0.05, N)
        hbr = sys_hbr + cereb_hbr + r.normal(0, 0.03, N)
        return hbo, hbr, cereb_hbo

    for (src, det, x, dist), pair in zip(short_specs + long_specs, short_pairs + long_pairs):
        is_short = pair in short_pairs
        glitch_gain = spatial_gain(x, 1.0, 0.2, 0.09) * 0.6
        hbo, hbr, cer = make_channel_conc(x, is_short, seed_ctr, glitch_gain); seed_ctr += 1
        true_conc[pair.label] = (hbo, hbr)
        true_cereb_hbo[pair.label] = cer

    def forward_raw(pair, hbo_uM, hbr_uM):
        abs_coef = np.array([ext(wl) for wl in pair.wavelengths])
        EL = abs_coef * pair.distance_m * PPF          # same EL as ChannelPair.set_extinction()
        conc = np.stack([hbo_uM, hbr_uM], axis=0) * 1e-3
        od = EL @ conc
        return 1.0 * 10**(-od)                          # raw intensity, baseline level = 1.0

    col = 0
    full = np.zeros((N, len(pairs)*2))
    for pair in pairs:
        hbo, hbr = true_conc[pair.label]
        raw = forward_raw(pair, hbo, hbr)
        for wi, wl in enumerate(pair.wavelengths):
            pair.idx_by_wl[wl] = col
            full[:, col] = raw[wi]
            col += 1

    return dict(t=t, full=full, pairs=pairs, short_pairs=short_pairs, long_pairs=long_pairs,
                cerebral=cerebral, bump_centers=bump_centers, true_cereb_hbo=true_cereb_hbo,
                cereb_period=cereb_period, glitch=glitch)


def run_pipeline(ds):
    '''Runs the *actual* production ChannelPair / ShortChannelArtifactRemover
    pipeline over the simulated raw-intensity stream, exactly as viewer.py /
    itbs_trigger.py do: update every pair, snapshot the long channels'
    pre-regression HbO, then let remover.process() overwrite them in place.'''
    pairs, long_pairs, short_pairs = ds["pairs"], ds["long_pairs"], ds["short_pairs"]
    full, t = ds["full"], ds["t"]
    remover = ShortChannelArtifactRemover(pairs, SHORT_CHANNEL_MAX_DISTANCE_M)

    tt, raw_hbo, clean_hbo, short_hbo = [], {p.label: [] for p in long_pairs}, {p.label: [] for p in long_pairs}, {p.label: [] for p in short_pairs}
    baseline_ready = False
    for i in range(full.shape[0]):
        sample, ti = full[i], t[i]
        if not baseline_ready:
            for p in pairs:
                p.add_baseline_sample(sample)
            if ti >= BASELINE_SEC:
                for p in pairs:
                    p.finalize_baseline()
                baseline_ready = True
            continue
        for p in pairs:
            p.update(sample)
        snap = {p.label: p.last_hbo for p in long_pairs}
        remover.process()
        tt.append(ti)
        for p in long_pairs:
            raw_hbo[p.label].append(snap[p.label]); clean_hbo[p.label].append(p.last_hbo)
        for p in short_pairs:
            short_hbo[p.label].append(p.last_hbo)

    tt = np.array(tt)
    for p in long_pairs:
        raw_hbo[p.label], clean_hbo[p.label] = np.array(raw_hbo[p.label]), np.array(clean_hbo[p.label])
    for p in short_pairs:
        short_hbo[p.label] = np.array(short_hbo[p.label])
    return remover, tt, raw_hbo, clean_hbo, short_hbo


ds = build_dataset()
print(f"Simulated {DURATION_SEC:g}s @ {SFREQ:g}Hz: {len(ds['short_pairs'])} short + {len(ds['long_pairs'])} long channels, "
      f"{len(ds['bump_centers'])} cerebral bumps @ ~{ds['cereb_period']:.2f}s spacing ({1/ds['cereb_period']:.3f} Hz).")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 2.8))
for i, p in enumerate(ds["pairs"]):
    is_short = p in ds["short_pairs"]
    ax.plot([p.src_pos[0]*1000, p.det_pos[0]*1000], [0, 0], color=SHORT_COLOR if is_short else LONG_COLOR,
            lw=6, alpha=0.6, solid_capstyle="round")
    y_label = 0.16 if i % 2 == 0 else 0.32
    ax.annotate(p.label, xy=(p.mid[0]*1000, y_label), ha="center", fontsize=10)
ax.set_yticks([]); ax.set_xlabel("probe position (mm)")
ax.set_title("Simulated montage: short channels (blue) vs long channels (orange)")
ax.set_ylim(-0.3, 0.55)
plt.tight_layout(); plt.show()

## 2. Running the production pipeline

The cell below drives the simulated stream through the exact same
baseline-then-update-then-regress sequence `viewer.py`/`itbs_trigger.py` use,
so `raw_hbo` is what the trigger loop would have seen *without* this
project's new artifact-removal step, and `clean_hbo` is what it sees now.

In [ ]:
remover, tt, raw_hbo, clean_hbo, short_hbo = run_pipeline(ds)
long_pairs, short_pairs = ds["long_pairs"], ds["short_pairs"]
print(f"remover.active={remover.active}  ({len(remover.short_pairs)} short / {len(remover.long_pairs)} long)")
for lp in remover.long_pairs:
    print(f"  {lp.label}: nearest short channel(s) = {[sp.label for sp in remover._nearest[lp.label]]}")

true_group = -np.mean([ds["true_cereb_hbo"][p.label] for p in long_pairs], axis=0)[-len(tt):]  # noise-free ground truth
activity_raw = -np.mean([raw_hbo[p.label] for p in long_pairs], axis=0)     # activity = -mean(HbO), matches activity_core convention
activity_clean = -np.mean([clean_hbo[p.label] for p in long_pairs], axis=0)

print(f"\nGroup activity std over full recording:  raw={activity_raw.std():.3f} uM   cleaned={activity_clean.std():.3f} uM"
      f"   ({100*(1-activity_clean.std()/activity_raw.std()):.0f}% variance-domain reduction)")

In [ ]:
lp0 = long_pairs[0]
window = (tt > 200) & (tt < 260)
fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
axes[0].plot(tt[window], raw_hbo[lp0.label][window], color=LONG_COLOR, label="raw (pre-regression)")
axes[0].plot(tt[window], ds["true_cereb_hbo"][lp0.label][-len(tt):][window], color=TRUTH_COLOR, ls="--", label="true cerebral component")
axes[0].set_ylabel("HbO (uM)"); axes[0].set_title(f"{lp0.label}: raw"); axes[0].legend(loc="upper right")
axes[1].plot(tt[window], clean_hbo[lp0.label][window], color=CLEAN_COLOR, label="RLS-cleaned")
axes[1].plot(tt[window], ds["true_cereb_hbo"][lp0.label][-len(tt):][window], color=TRUTH_COLOR, ls="--", label="true cerebral component")
axes[1].set_ylabel("HbO (uM)"); axes[1].set_xlabel("time (s)"); axes[1].set_title(f"{lp0.label}: after short-channel RLS regression")
axes[1].legend(loc="upper right")
plt.tight_layout(); plt.show()

## 3. Periodicity: is the signal cleaner?

Two systemic bands overlap the frequencies that matter for this project:
respiration and the cardiac band are far from the trigger cadence, but the
**Mayer wave (~0.1 Hz) sits almost on top of the 0.1 Hz tone from
`STANDARD_BURST_INTERVAL_SEC=10s`** -- exactly the frequency an activity
trace's periodicity would need to be judged at. This makes the choice of
spectral resolution matter a lot for answering "is it cleaner": a coarse,
wide-band power comparison and a properly-resolved narrow-band SNR estimate
can give different answers, so both are shown below.

In [ ]:
def psd(x, seg_sec):
    return welch(x, fs=SFREQ, nperseg=int(seg_sec*SFREQ), noverlap=int(seg_sec*SFREQ*0.75))

f_coarse, p_raw_coarse = psd(activity_raw, 200)
_, p_clean_coarse = psd(activity_clean, 200)
f_fine, p_raw_fine = psd(activity_raw, 400)
_, p_clean_fine = psd(activity_clean, 400)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
for ax, f, praw, pclean, title in [
    (axes[0], f_coarse, p_raw_coarse, p_clean_coarse, "200s segments (coarse resolution)"),
    (axes[1], f_fine, p_raw_fine, p_clean_fine, "400s segments (fine resolution)"),
]:
    ax.semilogy(f, praw, color=LONG_COLOR, label="raw")
    ax.semilogy(f, pclean, color=CLEAN_COLOR, label="RLS-cleaned")
    ax.axvline(1/ds["cereb_period"], color=TRUTH_COLOR, ls="--", lw=1.5, label="cerebral/trigger-cadence tone")
    ax.axvspan(0.08, 0.13, color="gold", alpha=0.15, label="Mayer-wave band")
    ax.axvline(1.1, color="gray", ls=":", lw=1, label="cardiac")
    ax.axvline(0.25, color="gray", ls=":", lw=1)
    ax.set_xlim(0, 1.6); ax.set_xlabel("frequency (Hz)"); ax.set_title(title)
axes[0].set_ylabel("PSD (uM^2/Hz)"); axes[0].legend(fontsize=9, loc="upper right")
plt.suptitle("Group activity power spectrum: raw vs RLS-cleaned")
plt.tight_layout(); plt.show()

In [ ]:
def band_power(f, p, lo, hi):
    return p[(f >= lo) & (f <= hi)].sum()

def narrowband_snr(f, p, target, halfwidth_bins=2, flank_bins=4):
    idx = np.argmin(np.abs(f - target))
    flank = list(range(idx-flank_bins-halfwidth_bins, idx-halfwidth_bins)) + list(range(idx+halfwidth_bins+1, idx+halfwidth_bins+flank_bins+1))
    flank = [k for k in flank if 0 <= k < len(p)]
    return p[idx] / np.median(p[flank]), p[idx], np.median(p[flank])

rows = []
for label, lo, hi in [("Mayer band (0.07-0.13Hz)", 0.07, 0.13), ("Cardiac band (0.9-1.4Hz)", 0.9, 1.4), ("Respiration band (0.18-0.32Hz)", 0.18, 0.32)]:
    rows.append({"band": label, "raw power": band_power(f_coarse, p_raw_coarse, lo, hi),
                 "cleaned power": band_power(f_coarse, p_clean_coarse, lo, hi),
                 "reduction": f"{100*(1 - band_power(f_coarse, p_clean_coarse, lo, hi)/band_power(f_coarse, p_raw_coarse, lo, hi)):.0f}%"})
band_df = pd.DataFrame(rows)
print("Broadband power reduction after short-channel regression:")
display(band_df)

snr_raw_coarse, *_ = narrowband_snr(f_coarse, p_raw_coarse, 1/ds["cereb_period"])
snr_clean_coarse, *_ = narrowband_snr(f_coarse, p_clean_coarse, 1/ds["cereb_period"])
snr_raw_fine, peak_raw, floor_raw = narrowband_snr(f_fine, p_raw_fine, 1/ds["cereb_period"])
snr_clean_fine, peak_clean, floor_clean = narrowband_snr(f_fine, p_clean_fine, 1/ds["cereb_period"])

snr_df = pd.DataFrame([
    {"resolution": "coarse (200s segments, ~7 bins spanning the Mayer hump)", "SNR raw": f"{snr_raw_coarse:.1f}", "SNR cleaned": f"{snr_clean_coarse:.1f}"},
    {"resolution": "fine (400s segments, floor from immediate flanking bins only)", "SNR raw": f"{snr_raw_fine:.1f}", "SNR cleaned": f"{snr_clean_fine:.1f}"},
])
print("\nPeak-to-local-floor SNR at the trigger-cadence tone:")
display(snr_df)
print(f"\nAt fine resolution: peak power raw={peak_raw:.2f} -> cleaned={peak_clean:.2f} ({100*(1-peak_clean/peak_raw):.0f}% drop -- "
      f"some genuine attenuation, since the tone sits inside the Mayer band); "
      f"local noise floor raw={floor_raw:.3f} -> cleaned={floor_clean:.4f} ({100*(1-floor_clean/floor_raw):.0f}% drop -- "
      f"floor falls much faster than the peak, so net SNR improves by {snr_clean_fine/snr_raw_fine:.0f}x.")

quiet = (f_coarse > 0.5) & (f_coarse < 0.8)   # a band with no injected physiological content at all
quiet_raw, quiet_clean = p_raw_coarse[quiet].mean(), p_clean_coarse[quiet].mean()
print(f"\nA quiet band with no injected physiological content (0.5-0.8Hz) tells the other half of the story: "
      f"PSD there goes {quiet_raw:.2e} -> {quiet_clean:.2e} uM^2/Hz, i.e. RISES {quiet_clean/quiet_raw:.0f}x after cleaning. "
      f"This is real and expected, not a bug: an online adaptive filter cannot perfectly separate correlated systemic content "
      f"from target signal using a single broadband time-domain fit, so removing the strong correlated peaks leaves behind a small "
      f"amount of 'misadjustment' noise spread across all frequencies (a standard, well-characterized property of adaptive filters, "
      f"scaling with how fast the filter is allowed to adapt). It is a good trade here -- the quiet-band floor is still tiny in "
      f"absolute terms next to the peaks it replaced -- but it is the reason the forgetting factor sensitivity in Section 5 matters: "
      f"faster adaptation (smaller lambda) means more misadjustment noise, which is exactly why lowering lambda hurt baseline "
      f"forecasting there.")

**Reading the table.** At coarse resolution the two SNR numbers look
similar (or even favor "raw") because the comparison band is wide enough to
swallow the entire Mayer hump -- both peak and floor shrink together and the
ratio is noisy. At the resolution that actually isolates the trigger-cadence
bin from its immediate neighbours, the picture is unambiguous: the noise
floor collapses far more than the signal peak, so the **narrowband SNR
improves by roughly an order of magnitude**. The honest caveat is real,
though -- because the cerebral tone and the Mayer wave are close in
frequency, the regression does remove a real (if small) fraction of the
peak itself. That is an inherent, not a tuning, limitation: short-channel
regression separates signals by their *spatial* correlation with a
superficial reference, not by frequency, so it does not know that some of
the shared 0.1 Hz power in a long channel is genuinely cerebral. Given how
close `STANDARD_BURST_INTERVAL_SEC` (10s -> 0.1 Hz) sits to the Mayer band,
this project's own choice of cadence is exactly the case where that
trade-off is most favorable, since Mayer-band contamination is precisely
what would otherwise be hardest to tell apart from the trigger signal.

There is a second, opposite-signed caveat visible in the "400s" panel above:
away from any physiological band (e.g. 0.5-0.8 Hz, which has no injected
content at all) the cleaned trace's floor sits visibly *above* the raw
trace's. That is not a bug -- it is the signature of an online adaptive
filter's inherent **misadjustment noise**, quantified next.

## 4. Peak/trough detection quality

Feeding the raw vs cleaned activity trace through the actual
`ActivityAnalyzer` causal detector (as `itbs_trigger.py` does) and matching
detections against the known bump centers.

In [ ]:
def run_detector(activity):
    an = ActivityAnalyzer(SFREQ)
    detected = []
    for ti, v in zip(tt, activity):
        for ev in an.push(ti, v):
            if ev["kind"] == "trough":  # activity=-mean(HbO), so an HbO increase shows up as an activity *trough*
                detected.append(ev["t"])
    return np.array(detected)

def match_error(detected, true, tol=3.0):
    errs, used = [], set()
    for d in detected:
        cand = sorted((abs(d - x), i) for i, x in enumerate(true) if i not in used)
        if cand and cand[0][0] < tol:
            used.add(cand[0][1]); errs.append(d - true[cand[0][1]])
    return np.array(errs), len(errs), len(true) - len(errs), len(detected) - len(errs)

true_bumps = ds["bump_centers"][(ds["bump_centers"] > tt[0]) & (ds["bump_centers"] < tt[-1])]
det_raw, det_clean = run_detector(activity_raw), run_detector(activity_clean)
errs_raw, m_raw, miss_raw, fp_raw = match_error(det_raw, true_bumps)
errs_clean, m_clean, miss_clean, fp_clean = match_error(det_clean, true_bumps)

detect_df = pd.DataFrame([
    {"signal": "raw", "n_true_bumps": len(true_bumps), "matched": m_raw, "missed": miss_raw, "false_positives": fp_raw,
     "timing_jitter_std_s": f"{errs_raw.std():.2f}", "mean_offset_s": f"{errs_raw.mean():.2f}"},
    {"signal": "RLS-cleaned", "n_true_bumps": len(true_bumps), "matched": m_clean, "missed": miss_clean, "false_positives": fp_clean,
     "timing_jitter_std_s": f"{errs_clean.std():.2f}", "mean_offset_s": f"{errs_clean.mean():.2f}"},
])
display(detect_df)
print(f"\nFalse-positive triggers: {fp_raw} (raw) -> {fp_clean} (cleaned), a {100*(1-fp_clean/fp_raw):.0f}% reduction in spurious,"
      f" non-cerebral peak/trough detections -- i.e. fewer stimulator pulses `itbs_trigger.py` would fire on cardiac/Mayer noise alone.")
print("Timing jitter on *matched* detections is essentially unchanged: MIN_PEAK_DISTANCE_SEC (5s) already dominates the timing"
      " budget, so regression mainly helps by suppressing spurious noise-driven detections, not by sharpening genuine ones.")

## 5. Interaction with a phastimate-style AR forecasting stage

[Phastimate](https://github.com/bnplab/phastimate) (Zrenner et al., 2018,
*Brain Stimulation*) is the standard real-time method for phase-triggered
closed-loop EEG-TMS: fit a short autoregressive (AR) model on a trailing
window, forward-extrapolate a few samples to beat the group-delay of any
causal band-pass/Hilbert step, and trigger on the extrapolated phase. **This
repository does not currently use phastimate or an AR forecasting stage --**
`itbs_trigger.py` triggers on the causal peak/trough detector instead. The
question here is forward-looking: if an AR-forecasting stage were added on
top of the activity trace, would the new RLS artifact-removal stage help,
hurt, or interact badly with it?

A plain AR(`AR_ORDER`) model is fit by least squares on a trailing
`AR_WINDOW_SEC` window and iterated forward `AR_FORECAST_SEC` ahead --
structurally the same "fit-on-a-window, extrapolate forward" shape
phastimate uses for its own edge correction. Forecast quality is scored
against the **known, noise-free cerebral ground truth** (`true_group`,
available only because this is simulated data) via RMSE at the forecast
horizon.

In [ ]:
def fit_ar_ls(window, order):
    n = len(window)
    X = np.column_stack([window[order-1-k:n-1-k] for k in range(order)])
    y = window[order:]
    coeffs, *_ = np.linalg.lstsq(X, y, rcond=None)
    return coeffs

def forecast_ar(history_tail, coeffs, order, n_ahead):
    hist = list(history_tail[-order:])
    out = []
    for _ in range(n_ahead):
        pred = float(np.dot(coeffs, hist[-order:][::-1]))
        out.append(pred); hist.append(pred)
    return np.array(out)

AR_WINDOW, AR_HORIZON = int(AR_WINDOW_SEC*SFREQ), int(AR_FORECAST_SEC*SFREQ)

def eval_ar_forecasts(activity, true_signal, cut_indices, order=AR_ORDER, window=AR_WINDOW, horizon=AR_HORIZON):
    errs = []
    for ci in cut_indices:
        if ci - window < 0 or ci + horizon >= len(activity):
            continue
        coeffs = fit_ar_ls(activity[ci-window:ci], order)
        fc = forecast_ar(activity[ci-window:ci], coeffs, order, horizon)
        errs.append(np.sqrt(np.mean((fc - true_signal[ci:ci+horizon])**2)))
    return np.array(errs)

rng = np.random.default_rng(7)
cand = np.where((tt > 100) & (tt < DURATION_SEC - 100))[0]
cut_idx_baseline = rng.choice(cand, size=N_EVAL_CUTS_BASELINE, replace=False)

err_raw = eval_ar_forecasts(activity_raw, true_group, cut_idx_baseline)
err_clean = eval_ar_forecasts(activity_clean, true_group, cut_idx_baseline)
ar_df = pd.DataFrame([{"input": "raw", "forecast RMSE (uM)": f"{err_raw.mean():.3f} +/- {err_raw.std():.3f}"},
                       {"input": "RLS-cleaned", "forecast RMSE (uM)": f"{err_clean.mean():.3f} +/- {err_clean.std():.3f}"}])
print(f"{AR_FORECAST_SEC:g}s-ahead AR({AR_ORDER}) forecast error, {len(err_raw)} random causal cut points, "
      f"scored against the noise-free cerebral ground truth:")
display(ar_df)
print(f"-> {err_raw.mean()/err_clean.mean():.1f}x lower forecast RMSE on the cleaned signal: a plain AR model fit on raw activity "
      f"mostly ends up forecasting cardiac/Mayer oscillation, not the cerebral trend.")

### Robustness to a shared motion-like transient

The real interaction risk is not the RLS filter's *steady-state* behaviour
(scored above) but its *adaptation transient*: because it is a causal,
online filter, its weights update on every sample, so a sudden change in the
short-channel reference (e.g. a motion artifact) can momentarily perturb
exactly the newest samples an AR forward-extrapolation model leans on most.
To test this, a shared Gaussian transient (`GLITCH_AMP_UM`, `GLITCH_DUR`) is
injected into both a short channel and its coupled long channels around
`t=GLITCH_T` -- modelling a real optode-coupling motion event that shows up
in both -- and the forecast is re-scored at cut points straddling it.

In [ ]:
ds_g = build_dataset(inject_transient=True)
remover_g, tt_g, raw_hbo_g, clean_hbo_g, short_hbo_g = run_pipeline(ds_g)
long_pairs_g, short_pairs_g = ds_g["long_pairs"], ds_g["short_pairs"]
true_group_g = -np.mean([ds_g["true_cereb_hbo"][p.label] for p in long_pairs_g], axis=0)[-len(tt_g):]
activity_raw_g = -np.mean([raw_hbo_g[p.label] for p in long_pairs_g], axis=0)
activity_clean_g = -np.mean([clean_hbo_g[p.label] for p in long_pairs_g], axis=0)

glitch_region = (tt_g > GLITCH_T - 2) & (tt_g < GLITCH_T + 15)
cut_idx_glitch = np.where(glitch_region)[0]
baseline_region_g = (tt_g > 100) & (tt_g < GLITCH_T - 20)
cut_idx_baseline_g = rng.choice(np.where(baseline_region_g)[0], size=N_EVAL_CUTS_BASELINE, replace=False)

err_raw_b = eval_ar_forecasts(activity_raw_g, true_group_g, cut_idx_baseline_g)
err_clean_b = eval_ar_forecasts(activity_clean_g, true_group_g, cut_idx_baseline_g)
err_raw_g = eval_ar_forecasts(activity_raw_g, true_group_g, cut_idx_glitch)
err_clean_g = eval_ar_forecasts(activity_clean_g, true_group_g, cut_idx_glitch)

transient_df = pd.DataFrame([
    {"input": "raw", "baseline RMSE": f"{err_raw_b.mean():.3f}", "near-transient RMSE": f"{err_raw_g.mean():.3f}",
     "degradation": f"{100*(err_raw_g.mean()/err_raw_b.mean()-1):+.0f}%"},
    {"input": "RLS-cleaned", "baseline RMSE": f"{err_clean_b.mean():.3f}", "near-transient RMSE": f"{err_clean_g.mean():.3f}",
     "degradation": f"{100*(err_clean_g.mean()/err_clean_b.mean()-1):+.0f}%"},
])
display(transient_df)

fig, ax = plt.subplots(figsize=(11, 3.2))
w = (tt_g > GLITCH_T-20) & (tt_g < GLITCH_T+20)
ax.plot(tt_g[w], activity_raw_g[w], color=LONG_COLOR, label="raw", alpha=0.8)
ax.plot(tt_g[w], activity_clean_g[w], color=CLEAN_COLOR, label="RLS-cleaned")
ax.axvline(GLITCH_T, color="crimson", ls=":", label="injected motion-like transient")
ax.set_xlabel("time (s)"); ax.set_ylabel("group activity (uM)"); ax.legend(fontsize=9)
ax.set_title("Group activity around an injected shared short+long transient")
plt.tight_layout(); plt.show()

At the current default forgetting factor (`RLS_FORGETTING_FACTOR=0.999`),
the cleaned signal's forecast error is essentially **unchanged** near the
transient, while the raw signal's forecast (already worse) degrades
further. In this simulation there is **no evidence of a negative
interaction** between the RLS stage and a downstream AR-forecasting stage --
if anything, the RLS stage visibly absorbs the shared transient (see the
plot: the orange raw trace jumps, the green cleaned trace barely moves)
because the transient is, by construction, correlated with the short
channel it is regressed against, which is exactly the mechanism this
preprocessing step is designed to exploit.

**But the forgetting factor matters.** The finding above is specific to a
forgetting factor near 1 (slow adaptation, ~1000-sample/100s effective
memory at 10 Hz). Re-running the regression with faster-adapting filters
(smaller lambda) shows why the current default should not casually be
lowered "to track transients faster":

In [ ]:
def refilter_with_lambda(lam):
    out = {}
    for lp in remover_g.long_pairs:
        refs = remover_g._nearest[lp.label]
        f = RLSFilter(n_refs=len(refs), forgetting_factor=lam)
        x = np.column_stack([short_hbo_g[r.label] for r in refs])
        d = raw_hbo_g[lp.label]
        cleaned = np.empty_like(d)
        for i in range(len(d)):
            e, _ = f.update(x[i], d[i])
            cleaned[i] = e
        out[lp.label] = cleaned
    return -np.mean([out[p.label] for p in long_pairs_g], axis=0)

lam_rows = []
for lam in [0.999, 0.99, 0.95]:
    act = refilter_with_lambda(lam)
    eb = eval_ar_forecasts(act, true_group_g, cut_idx_baseline_g)
    eg = eval_ar_forecasts(act, true_group_g, cut_idx_glitch)
    lam_rows.append({"forgetting factor (lambda)": lam, "baseline RMSE": f"{eb.mean():.3f}",
                      "near-transient RMSE": f"{eg.mean():.3f}"})
lam_df = pd.DataFrame(lam_rows)
display(lam_df)
print("Lowering lambda does not meaningfully improve near-transient forecasting, and it substantially *worsens* baseline "
      "forecasting -- a faster-adapting filter chases short-term noise instead of tracking the slowly-varying physiological "
      "coupling, inflating its own output variance. Pushed further (lambda ~0.9) the RLS covariance update (P matrix) can "
      "become numerically ill-conditioned and diverge. Recommendation: leave RLS_FORGETTING_FACTOR close to 1 (the current "
      "0.999 default); if faster tracking of coupling changes is ever needed, add an explicit outlier/transient gate rather "
      "than globally lowering lambda.")

## 6. Should the confound regression be done inside the AR model itself?

Two designs are compared for a hypothetical AR/phastimate-style stage:

- **Cascade (current design's natural extension):** the existing
  `ShortChannelArtifactRemover` regresses out the short-channel signal
  first (a dedicated, low-dimensional -- up to
  `N_NEAREST_SHORT_CHANNELS`-reference -- regression per long channel), then
  a plain AR model is fit/forecast on the cleaned signal, as in Section 5.
- **Joint ARX:** a single model that regresses the target on *both* its own
  AR lags *and* the short-channel exogenous input simultaneously (an
  AutoRegressive-with-eXogenous-input model), fit by least squares on the
  same trailing window for a fair, apples-to-apples comparison against the
  cascade's own windowed AR refit.

In [ ]:
def fit_arx_ls(y_window, x_window, order):
    n = len(y_window)
    X_ar = np.column_stack([y_window[order-1-k:n-1-k] for k in range(order)])
    X_exog = x_window[order:].reshape(-1, 1)
    X = np.column_stack([X_ar, X_exog])
    coeffs, *_ = np.linalg.lstsq(X, y_window[order:], rcond=None)
    return coeffs[:order], coeffs[order:]

def forecast_arx(y_tail, x_last, a_coeffs, b_coeffs, order, n_ahead):
    hist = list(y_tail[-order:])
    out = []
    for _ in range(n_ahead):
        pred = float(np.dot(a_coeffs, hist[-order:][::-1]) + b_coeffs[0]*x_last)
        out.append(pred); hist.append(pred)
    return np.array(out)

def eval_arx(y, x, true_signal, cut_indices, order=AR_ORDER, window=AR_WINDOW, horizon=AR_HORIZON):
    errs = []
    for ci in cut_indices:
        if ci - window < 0 or ci + horizon >= len(y):
            continue
        a, b = fit_arx_ls(y[ci-window:ci], x[ci-window:ci], order)
        fc = forecast_arx(y[ci-window:ci], x[ci-1], a, b, order, horizon)
        errs.append(np.sqrt(np.mean((fc - true_signal[ci:ci+horizon])**2)))
    return np.array(errs)

exog = -np.mean([short_hbo_g[p.label] for p in short_pairs_g], axis=0)  # pooled short-channel exogenous input

e_raw_b = eval_ar_forecasts(activity_raw_g, true_group_g, cut_idx_baseline_g)
e_raw_g = eval_ar_forecasts(activity_raw_g, true_group_g, cut_idx_glitch)
e_cascade_b = eval_ar_forecasts(activity_clean_g, true_group_g, cut_idx_baseline_g)
e_cascade_g = eval_ar_forecasts(activity_clean_g, true_group_g, cut_idx_glitch)
e_arx_b = eval_arx(activity_raw_g, exog, true_group_g, cut_idx_baseline_g)
e_arx_g = eval_arx(activity_raw_g, exog, true_group_g, cut_idx_glitch)

joint_df = pd.DataFrame([
    {"design": "raw, plain AR (no regression)", "baseline RMSE": f"{e_raw_b.mean():.3f}", "near-transient RMSE": f"{e_raw_g.mean():.3f}"},
    {"design": "cascade: ShortChannelArtifactRemover -> AR", "baseline RMSE": f"{e_cascade_b.mean():.3f}", "near-transient RMSE": f"{e_cascade_g.mean():.3f}"},
    {"design": "joint ARX (regression folded into the AR model)", "baseline RMSE": f"{e_arx_b.mean():.3f}", "near-transient RMSE": f"{e_arx_g.mean():.3f}"},
])
display(joint_df)

**The cascade wins clearly, in both conditions.** The joint ARX model
performs barely better than doing no regression at all. This is not simply
"joint is worse in principle" -- it reflects a real, structural mismatch: the
dedicated remover gives each long channel its own low-dimensional (1-3
reference) regression with a slow, near-unity forgetting factor tuned for
slowly-varying physiological coupling, refit continuously online. Folding
that into a single windowed ARX least-squares fit forces one model and one
window length to serve two very different jobs at once -- tracking
(slow) systemic coupling and fitting (locally-refit) oscillatory AR
dynamics -- and a single pooled zero-lag exogenous term is also a much
coarser regressor than the remover's per-channel nearest-neighbour
selection. A richer joint design (per-channel exogenous references, several
exogenous lags, separately-tuned regularization) might close some of this
gap, but the evidence here does not support moving confound regression
*into* the AR stage as it stands -- **keep it as a dedicated preprocessing
stage**, feeding its output into whatever forecasting/phase-estimation model
comes next.

## Summary

1. **Cleaner periodicity: yes, but resolution-dependent and not free.**
   Broadband systemic power (cardiac, respiration, Mayer) drops sharply
   after regression, and the trigger-cadence tone's *narrowband* SNR
   improves by roughly an order of magnitude once measured at adequate
   frequency resolution -- a coarse wide-band comparison can look
   ambiguous or even favor "raw" purely from bin-averaging artifacts.
   Because `STANDARD_BURST_INTERVAL_SEC` (10s, 0.1 Hz) sits inside the
   Mayer-wave band, this project's chosen cadence is exactly the case where
   that contamination -- and this fix -- matters most. The cost is real
   too: some of the genuine signal at that frequency is inevitably
   attenuated along with the Mayer-band noise, since the regression
   separates by spatial (short-vs-long channel), not spectral, structure.
   Downstream, `ActivityAnalyzer` mainly benefits by suppressing spurious
   noise-driven triggers (~28% fewer false detections here), not by
   sharpening the timing of genuine ones. There is also an opposite-signed
   cost worth remembering: the regression measurably *raises* the noise
   floor in frequency bands that had no systemic contamination to begin
   with (an adaptive filter's inherent misadjustment noise) -- a real trade,
   not a free lunch, though a clearly favorable one here since that floor
   stays tiny next to the peaks it replaces.

2. **No negative interaction found with an AR/phastimate-style stage at the
   current default settings** -- forecasting on the cleaned signal is both
   more accurate and remains stable through an injected motion-like
   transient shared between short and long channels. The one real risk
   surfaced here is **tuning the forgetting factor too aggressively**:
   lowering it does not improve transient robustness and instead degrades
   baseline forecasting (consistent with the misadjustment-noise mechanism
   above -- faster adaptation means more of it), and can destabilize the RLS
   covariance update entirely. Keep `RLS_FORGETTING_FACTOR` close to 1
   (0.999, current default); if this pipeline later needs an
   AR/phase-forecasting stage, this analysis found no reason to change that
   value for its sake.

3. **Keep confound regression as a separate preprocessing stage.** A joint
   ARX model that folds short-channel regression into the AR fit performed
   little better than no regression at all in this test -- the dedicated,
   per-channel, slowly-adapting `ShortChannelArtifactRemover` clearly
   outperformed collapsing both jobs into one filter with one timescale.

**Caveats.** All quantitative numbers above are from a synthetic dataset
with a chosen (if physiologically-motivated) coupling structure, noise
level, and cadence -- they establish *mechanism and direction of effect*,
not universal magnitudes. Re-running this notebook with `REAL_SNIRF_PATH`
set to a real recording would be needed to get numbers specific to any
particular subject/session; the code above (`build_dataset`/`run_pipeline`)
is structured so that swapping in a real stream only requires replacing the
`build_dataset()` call with one that parses real channel metadata and raw
samples the same way `stream_snirf_lsl.py`/`activity_core.parse_channels`
do -- the pipeline, detector, and forecasting code are unchanged.